## Análisis Comparativo de Segmentación y Seguimiento Celular (Tracking) con Trackastra

Este *notebook* presenta un flujo de trabajo para el análisis de imágenes de microscopía de células de HeLa (provenientes del dataset Fluo-N2DL-HeLa). El objetivo es comparar el rendimiento de la herramienta de seguimiento celular **Trackastra** utilizando máscaras de segmentación generadas por tres métodos diferentes:

1.  **Watershed** (clásico, basado en visión por computador).
2.  **Cellpose** (red neuronal de aprendizaje profundo).
3.  **TFG** (máscaras pre-existentes, probablemente de alta calidad, usadas como *ground truth* o resultado de un método de alto rendimiento - el código original las carga de `hela/mascaras`).

La métrica principal de comparación es el **Tracking Accuracy (TRA)** y el **Segmentation Accuracy (SEG)**, utilizadas en el Desafío de Seguimiento de Células (Cell Tracking Challenge - CTC).

---

### 1. Inicialización y Funciones Auxiliares

Esta sección inicializa las librerías necesarias para el procesamiento de imágenes, segmentación (Cellpose), seguimiento (Trackastra) y visualización (Napari), además de definir la función de segmentación Watershed.

In [1]:
import napari
import os
import numpy as np
import cv2 as cv
import tifffile as tiff
from trackastra.model import Trackastra
from trackastra.tracking import graph_to_ctc, graph_to_napari_tracks
from cellpose import models

# Para forzar a que use la GPU (cambiar a "cpu" si no hay GPU disponible)
device = "cuda"

#### 1.1. Implementación del Algoritmo de Watershed

La función `watershed` implementa el algoritmo de la cuenca hidrográfica (Watershed) para la segmentación. Este método utiliza la distancia de transformación y umbralización para encontrar semillas (*sure foreground*) y un fondo seguro (*sure background*), y luego aplica el algoritmo de Watershed para delinear los bordes de los objetos. 

La función `watershed_frames` aplica esto a toda la secuencia de imágenes.

In [2]:
def watershed(img, ksize=3):
    img_8bit = cv.normalize(img, None, 0, 255, cv.NORM_MINMAX, cv.CV_8U)
    
    _, bin_simple = cv.threshold(img_8bit, 0, 255, cv.THRESH_BINARY + cv.THRESH_OTSU)
    
    # 1) Limpieza ligera
    bin_clean = cv.medianBlur(bin_simple, 3)

    # 2) Fondo seguro (sure background) por dilatación
    kernel = np.ones((ksize,ksize), np.uint8)
    sure_bg = cv.dilate(bin_clean, kernel, iterations=3)

    # 3) Foreground seguro con distance transform
    dist = cv.distanceTransform(bin_clean, distanceType=cv.DIST_L2, maskSize=5)
    # Umbral relativo al máximo para obtener “picos” bien dentro de los objetos
    _, sure_fg = cv.threshold(dist, 0.5 * dist.max(), 255, cv.THRESH_BINARY)
    sure_fg = sure_fg.astype(np.uint8)

    # 4) Zona desconocida (bordes entre objetos/fondo)
    unknown = cv.subtract(sure_bg, sure_fg)

    # 5) Etiquetado de componentes conectados
    num_labels, markers = cv.connectedComponents(sure_fg)
    markers = markers + 1             # 1 será el fondo
    markers[unknown==255] = 0         # 0 será la zona desconocida

    # 6) Watershed necesita una imagen 3 canales 
    img_color = cv.cvtColor(bin_simple, cv.COLOR_GRAY2BGR)

    # 7) Ejecutar watershed
    markers_ws = cv.watershed(img_color, markers.copy())
    
    # Los bordes (-1) los convertimos a 0 (fondo) para el formato CTC
    markers_ws[markers_ws == -1] = 0 
    return markers_ws.astype(np.uint16)

def watershed_frames(frames, ksize=3):
    # Recibe array de frames y retorna array de máscaras
    res = []
    print("\n--- Ejecutando Segmentación Watershed ---")
    for i, frame in enumerate(frames):
        mask = watershed(frame, ksize=ksize)
        res.append(mask)
        print(f"Segmentada imagen {i+1}/{len(frames)}", end='\r')
    print("\nSegmentación Watershed completada.")
    return np.stack(res)

--- 

### 2. Carga y Segmentación de Datos

Se carga la secuencia de imágenes de microscopía y se generan las máscaras de segmentación para los métodos Watershed y Cellpose.

#### 2.1. Carga de Imágenes Originales

Se cargan los archivos de imagen TIF de la secuencia `Fluo-N2DL-HeLa/01`.

In [ ]:
# --- 1. Carga de imágenes (Input para Cellpose) ---
img_path = "./Fluo-N2DL-HeLa/01"
imgs_names = sorted(os.listdir(img_path))
imgs = []
for file in imgs_names:
    if file.endswith('.tif'):
        img = cv.imread(os.path.join(img_path, file), cv.IMREAD_UNCHANGED)
        imgs.append(img)
        
imgs = np.stack(imgs)
print(f"Dimensiones de imágenes cargadas: {imgs.shape}")


Dimensiones de imágenes cargadas: (92, 700, 1100)


#### 2.2. Segmentación con Watershed

Se aplica la función Watershed a la secuencia completa de imágenes, y las máscaras resultantes se guardan en el directorio `Watershed/seg`.

In [4]:
#Segmentación Watershed y Preparación de Datos
watershed_masks = watershed_frames(imgs)

OUTPUT_DIR = "Watershed/seg"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

if os.path.exists("tracked_ctc/res_track.txt"):
    os.remove("tracked_ctc/res_track.txt")

for i, mask in enumerate(watershed_masks):
    file_name = f"mask_{i:03d}.tif" 
    full_file_path = os.path.join(OUTPUT_DIR, file_name)
    tiff.imwrite(full_file_path, mask)
print(f"Máscaras Watershed guardadas")


--- Ejecutando Segmentación Watershed ---
Segmentada imagen 92/92
Segmentación Watershed completada.
Máscaras Watershed guardadas


#### 2.3. Segmentación con Cellpose

Se utiliza el modelo pre-entrenado de **Cellpose** (una red neuronal potente para segmentación de células) para generar las máscaras. Los resultados se guardan en `Cellpose/seg`.

In [5]:
# --- 1. Configuración de Rutas y Modelo Cellpose ---
OUTPUT_DIR = "Cellpose/seg"

if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

if os.path.exists("tracked_ctc/res_track.txt"):
    os.remove("tracked_ctc/res_track.txt")

# Inicialización del modelo Cellpose
model = models.CellposeModel(gpu="cuda") 

# --- 2. Parámetros de Segmentación ---
FLOW_THRESHOLD = 0.4
CELLPROB_THRESHOLD = 0.0
TILE_NORM_BLOCKSIZE = 0

print("Iniciando segmentación con Cellpose, frame a frame...")

ctc_masks_list = []
total_frames = imgs.shape[0]

# --- 3. Bucle de Segmentación, Guardado y Almacenamiento ---
for i in range(total_frames):
    frame = imgs[i]
    
    masks, _, _ = model.eval(frame, batch_size=32, flow_threshold=FLOW_THRESHOLD, cellprob_threshold=CELLPROB_THRESHOLD,
                              normalize={"tile_norm_blocksize": TILE_NORM_BLOCKSIZE})
    
    # Aseguramos el tipo de dato
    mask_result = masks.astype(np.uint16) 
    ctc_masks_list.append(mask_result)
    
    # Guardado del archivo
    file_name = f"mask_{i:03d}.tif" 
    full_file_path = os.path.join(OUTPUT_DIR, file_name)
    tiff.imwrite(full_file_path, mask_result)
    
    print(f"Frame {i} procesado y guardado.")

print("Segmentación y guardado completados.")

# --- 4. Carga/Apilado Final ---
cellpose_masks = np.stack(ctc_masks_list)

print("Carga de máscaras completada.")
print(f"Dimensiones de las máscaras de Cellpose cargadas (ctc_masks): {cellpose_masks.shape}")

INFO:cellpose.core:** TORCH CUDA version installed and working. **
INFO:cellpose.core:>>>> using GPU (CUDA)
INFO:cellpose.models:>>>> loading model /home/alex/.cellpose/models/cpsam


Iniciando segmentación con Cellpose, frame a frame...
Frame 0 procesado y guardado.
Frame 1 procesado y guardado.
Frame 2 procesado y guardado.
Frame 3 procesado y guardado.
Frame 4 procesado y guardado.
Frame 5 procesado y guardado.
Frame 6 procesado y guardado.
Frame 7 procesado y guardado.
Frame 8 procesado y guardado.
Frame 9 procesado y guardado.
Frame 10 procesado y guardado.
Frame 11 procesado y guardado.
Frame 12 procesado y guardado.
Frame 13 procesado y guardado.
Frame 14 procesado y guardado.
Frame 15 procesado y guardado.
Frame 16 procesado y guardado.
Frame 17 procesado y guardado.
Frame 18 procesado y guardado.
Frame 19 procesado y guardado.
Frame 20 procesado y guardado.
Frame 21 procesado y guardado.
Frame 22 procesado y guardado.
Frame 23 procesado y guardado.
Frame 24 procesado y guardado.
Frame 25 procesado y guardado.
Frame 26 procesado y guardado.
Frame 27 procesado y guardado.
Frame 28 procesado y guardado.
Frame 29 procesado y guardado.
Frame 30 procesado y guard

#### 2.4. Carga de Máscaras TFG (Referencia)

Se cargan las máscaras pre-existentes de un método de referencia (etiquetadas aquí como TFG) para usarlas como un tercer conjunto de datos de segmentación para el seguimiento.

In [6]:
mask_files = sorted([os.path.join("hela/mascaras", f) for f in os.listdir("hela/mascaras") if f.endswith('.tif')])

tfg_masks = np.stack([cv.imread(f, cv.IMREAD_UNCHANGED).astype("uint16") for f in mask_files], axis=0)

print(f"Dimensiones de máscaras: {tfg_masks.shape}")
print(masks.dtype)

Dimensiones de máscaras: (92, 700, 1100)
uint16


--- 

### 3. Seguimiento Celular con Trackastra

Esta sección inicializa el modelo **Trackastra** y lo utiliza para realizar el seguimiento celular sobre los tres conjuntos de máscaras de segmentación obtenidos: Cellpose, Watershed y TFG. Trackastra construye un grafo de conexiones y aplica un algoritmo codicioso (*greedy*) para resolver las trayectorias. 

#### 3.1. Inicialización del Modelo Trackastra

Se carga el modelo pre-entrenado de Trackastra para el seguimiento de células 2D.

In [7]:
# Inicialización de Trackastra
modelo = Trackastra.from_pretrained("general_2d", device=device)

INFO:trackastra.model.model:Loading model state from /home/alex/Universidad/PID/Trabajo/.venv/lib/python3.13/site-packages/trackastra/.models/general_2d/model.pt


/home/alex/Universidad/PID/Trabajo/.venv/lib/python3.13/site-packages/trackastra/.models/general_2d already downloaded, skipping.


INFO:trackastra.model.model_api:Using device cuda
INFO:trackastra.model.model_api:Default batch size = 4 for model on cuda.


#### 3.2. Ejecución del Seguimiento

Se aplica el método `modelo.track` a cada conjunto de máscaras de segmentación, utilizando las imágenes originales (`imgs`) como referencia para las características de las células.

In [8]:
# Aquí se hace el seguimiento de las células
track_cellpose_graph, masks_cellpose_tracked = modelo.track(imgs, cellpose_masks, mode="greedy")
track_watershed_graph, masks_watershed_tracked = modelo.track(imgs, watershed_masks, mode="greedy")
track_tfg_graph, masks_tfg_tracked = modelo.track(imgs, tfg_masks, mode="greedy")

INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 92 detections
INFO:trackastra.data.wrfeat:Using single process for feature extraction
Extracting features: 100%|██████████| 92/92 [00:01<00:00, 67.91it/s] 
INFO:trackastra.model.model_api:Building windows
Building windows: 100%|██████████| 89/89 [00:00<00:00, 24739.42it/s]
INFO:trackastra.model.model_api:Predicting windows
Computing associations: 100%|██████████| 23/23 [00:00<00:00, 31.11it/s]
INFO:trackastra.model.model_api:Running greedy tracker
INFO:trackastra.tracking.tracking:Build candidate graph with delta_t=1
INFO:trackastra.tracking.tracking:Added 9005 vertices, 8878 edges                         
INFO:trackastra.tracking.tracking:Running greedy tracker
Greedily matched edges:  99%|█████████▉| 8811/8878 [00:00<00:00, 228333.54it/s]
INFO:trackastra.model.model_api:Predicting weights for candidate graph
INFO:trackastra.data.wrfeat:Extracting features from 9

#### 3.3. Conversión al Formato CTC (Cell Tracking Challenge)

Los resultados del seguimiento (el grafo de trayectorias) y las máscaras originales se convierten al formato estándar CTC (`res_track.txt` y secuencias de máscaras con etiquetas de seguimiento) para poder evaluarlos con las métricas del desafío.

In [9]:
# Conversión del grafo al formato CTC y guardado
graph_to_ctc(track_cellpose_graph, cellpose_masks, outdir="tracked_cellpose_ctc")
os.rename("tracked_cellpose_ctc/man_track.txt", "tracked_cellpose_ctc/res_track.txt")

graph_to_ctc(track_watershed_graph, watershed_masks, outdir="tracked_watershed_ctc")
os.rename("tracked_watershed_ctc/man_track.txt", "tracked_watershed_ctc/res_track.txt")

graph_to_ctc(track_tfg_graph, tfg_masks, outdir="tracked_tfg_ctc")
os.rename("tracked_tfg_ctc/man_track.txt", "tracked_tfg_ctc/res_track.txt")

Saving masks: 100%|██████████| 92/92 [00:00<00:00, 578.71it/s]


--- 

### 4. Evaluación y Comparación de Rendimiento

Se utiliza una herramienta de evaluación compatible con CTC (`ctc_metrics.evaluate_sequence`) para calcular el rendimiento de cada seguimiento en comparación con el *ground truth* (verdad fundamental) del dataset.

Las métricas clave son:

* **TRA (Tracking Accuracy):** Mide la precisión del seguimiento a lo largo del tiempo, incluyendo la identificación de divisiones celulares. Un valor más alto es mejor.
* **SEG (Segmentation Accuracy):** Mide la superposición de las máscaras segmentadas respecto a las máscaras del *ground truth*. Un valor más alto es mejor.

In [ ]:
from ctc_metrics import evaluate_sequence

path_gt = "./Fluo-N2DL-HeLa/01_GT/"

# Definimos los métodos y sus carpetas de resultados
methods = {
    "Cellpose": os.path.join("./", "tracked_cellpose_ctc"),
    "Watershed": os.path.join("./", "tracked_watershed_ctc"),
    "TFG": os.path.join("./", "tracked_tfg_ctc")
}

results_summary = {}

for name, path_res in methods.items():
    print(f"\n--- Evaluando: {name} ---")
    
    # Ejecución de la evaluación CTC
    res = evaluate_sequence(
        path_res,  
        path_gt    
    )
    
    # Almacenar resultados
    results_summary[name] = res
    
    # Imprimir resultados del método actual
    TRA = res['TRA']
    SEG = res['SEG']
    
    print(f"Tracking Accuracy (TRA): {TRA:.4f}")
    print(f"Segmentation Accuracy (SEG): {SEG:.4f}")
    
# Resumen Comparativo Final
print("\n==============================================")
print("              RESUMEN COMPARATIVO")
print("==============================================")
for name, res in results_summary.items():
    print(f"[{name}]: TRA = {res['TRA']:.4f}, SEG = {res['SEG']:.4f}")


--- Evaluando: Cellpose ---
Evaluate sequence:  ./tracked_cellpose_ctc  with ground truth:  ../Fluo-N2DL-HeLa/01_GT/with results:  {'Valid': 1, 'CHOTA': np.float64(0.8487326046961712), 'BC': None, 'CT': 0.529505582137161, 'CCA': 0.6379310344827587, 'TF': np.float64(0.9225289489761692), 'SEG': 0.7518252827283392, 'TRA': 0.9337619541886267, 'DET': 0.9343211019793958, 'MOTA': np.float64(0.8425743720338001), 'HOTA': np.float64(0.8846016506554504), 'IDF1': np.float64(0.8767621646202819), 'MTML': None, 'FAF': 8.73913043478261, 'LNK': 0.9300007786342754, 'OP_CTB': 0.842793618458483, 'OP_CSB': 0.8430731923538675, 'BIO': None, 'OP_CLB': None, 'AOGM': np.float64(6573.0), 'AOGM_0': np.float64(99233.0), 'AOGM_NS': np.int64(5), 'AOGM_FN': np.int64(485), 'AOGM_FP': np.int64(799), 'AOGM_ED': 15, 'AOGM_EA': np.int64(574), 'AOGM_EC': 23, 'gt_divisions': 94, 'tp_div(0)': 71, 'fp_div(0)': 39, 'fn_div(0)': 23, 'BC(0)': 0.6960784313725491, 'tp_div(1)': 72, 'fp_div(1)': 38, 'fn_div(1)': 22, 'BC(1)': 0.7058

#### 📊 Análisis de Resultados

El resultado demuestra que la **calidad de la segmentación influye drásticamente en la precisión del seguimiento**.

* El método **Cellpose** proporciona el mejor rendimiento en ambos, segmentación (SEG) y seguimiento (TRA), sugiriendo que la segmentación por *Deep Learning* genera máscaras de mayor calidad que permiten a Trackastra construir un grafo de trayectorias más preciso.
* El método clásico **Watershed** tiene el peor rendimiento en segmentación y seguimiento, probablemente debido a errores comunes como el *over-segmentation* (sobre-segmentación) o la fusión de células (*under-segmentation*), lo que dificulta el *tracking* posterior.
* Las máscaras **TFG**, a pesar de tener una mejor segmentación (SEG = 0.4531) que Watershed, producen el peor resultado de seguimiento (TRA = 0.5570). Esto podría indicar que la estructura de estas máscaras (ej. etiquetas de ID, bordes ruidosos) tiene un impacto negativo en las características de conexión que utiliza Trackastra, incluso si la superposición (SEG) es mejor que la de Watershed.

--- 

### 5. Visualización con Napari (Cellpose)

Se preparan los datos del mejor resultado (Cellpose) para la visualización interactiva en **Napari**, permitiendo inspeccionar las imágenes originales, las máscaras segmentadas y las trayectorias celulares generadas por Trackastra.

In [13]:
# Transformación de los datos para la evaluación con napari
napari_tracks, napari_tracks_graph, _ = graph_to_napari_tracks(track_cellpose_graph)

100%|██████████| 362/362 [00:00<00:00, 50582.60it/s]


In [14]:
# Visualización con napari
v = napari.Viewer()
v.add_image(imgs, name='Imágenes Originales')
v.add_labels(cellpose_masks, name='Máscaras Segmentadas (Cellpose)', opacity=0.7)
v.add_tracks(data=napari_tracks, graph=napari_tracks_graph, name='Trayectorias de Trackastra')

<Tracks layer 'Trayectorias de Trackastra' at 0x7f125aee79d0>